# Agentic AI for Sports Analytics — Experiments

This notebook demonstrates the full agentic pipeline for **tabular sports analytics**.
The system is designed for **coaches, performance analysts, scouts, and federation staff** — users with deep sports domain knowledge but limited machine learning expertise.

## What this system does
1. **Profiles** the dataset and runs **exploratory clustering** to find natural groupings (e.g. player archetypes, injury-risk tiers).
2. **Detects data quality issues** and, when necessary, **asks the user a plain-language clarification question** (e.g. "Is this column recorded before or after the match?").
3. **Proposes and evaluates** multiple preprocessing + modelling strategies via the Claude API.
4. **Generates example visualisations** (missingness heatmap, feature importance, confusion matrix, cluster projection).
5. **Builds a plain-language report** explaining findings and decisions.

## Recommended sports datasets
Replace `DATA_PATH` and `TARGET` with any of the following publicly available datasets:

| Dataset | Source | Suggested target | Task |
|---------|--------|-----------------|------|
| StatsBomb Open Data (match events) | [github.com/statsbomb/open-data](https://github.com/statsbomb/open-data) | `result` | Binary classification |
| NBA shot logs | [Kaggle](https://www.kaggle.com/dansbecker/nba-shot-logs) | `SHOT_RESULT` | Binary classification |
| FIFA 22 player attributes | [Kaggle](https://www.kaggle.com/datasets/stefanoleone992/fifa-22-complete-player-dataset) | `overall` | Regression |
| Injury/wellness data | Various | `injured` | Binary classification (severe imbalance) |

The demo below uses **`data/titanic.csv`** (Titanic survival) as a substitute — the system works identically with any tabular CSV.

**Prerequisites:** add your Anthropic API key to the `.env` file in the project root.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv()

print(f"Project root : {PROJECT_ROOT}")
print(f"SQLite store : {PROJECT_ROOT / 'storage' / 'runs.db'}")
print(f"ChromaDB dir : {PROJECT_ROOT / 'storage' / 'chroma_db'}")

In [ ]:
from src.data.loader import load_data, split_data
from src.agents.profiler import generate_profile, discover_clusters, summarize_patterns
from src.agents.issue_detector import detect_issues, request_user_clarification
from src.agents.evaluator import generate_visualisations, build_user_report
from src.orchestrator import run_agentic_pipeline

## 1. Load dataset

Replace `DATA_PATH` and `TARGET` with your sports dataset.

In [ ]:
DATA_PATH = "data/titanic.csv"   # ← replace with your sports dataset path
TARGET    = "Survived"           # ← replace with your target column

df = load_data(DATA_PATH)
print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

## 2. Profile + exploratory analysis

The Profiler Agent scans every column and runs **unsupervised k-means clustering** on the numeric features to identify natural groupings in the data.  In a sports context this might reveal player archetypes, match intensity tiers, or injury-risk profiles.

In [ ]:
profile = generate_profile(df, TARGET, run_clustering=True)

print(f"Task type : {profile.target_type.value}")
print(f"Rows      : {profile.n_rows:,}  |  Columns: {profile.n_cols}  |  Duplicates: {profile.n_duplicates}")

if profile.class_distribution:
    print(f"Classes   : {profile.class_distribution}")

# --- Exploratory cluster patterns ---
if profile.clusters:
    cl = profile.clusters
    print(f"\n─── Exploratory Clusters ────────────────────────────")
    print(f"  {cl.n_clusters} natural groups (silhouette={cl.silhouette_score:.3f}, DB={cl.davies_bouldin_index:.3f})")
    print(f"  Features used: {', '.join(cl.numeric_columns[:6])}" + (" ..." if len(cl.numeric_columns) > 6 else ""))
    print()
    for cid, summary in cl.cluster_summaries.items():
        print(f"  {summary}")
else:
    print("\nClustering: not enough numeric data for meaningful clusters.")

## 3. Detect data quality issues + clarification questions

The Issue Detector scans for missing values, outliers, class imbalance, and potential data leakage.  For issues that require domain knowledge to resolve, it formulates **plain-language questions** for the sports user — e.g. whether a column records a pre- or post-event measurement.

In [ ]:
issues = detect_issues(profile, df)

print(f"Detected issues ({len(issues)}):")
for issue in issues:
    print(f"  [{issue.severity.value.upper():<6}] {issue.issue_type.value:<22}  {issue.description}")

# --- Clarification questions ---
questions = request_user_clarification(issues, profile)
if questions:
    print(f"\n─── Clarification Questions ({len(questions)}) ────────────────────")
    for q in questions:
        print(f"\n  [{q.issue_type.upper()}] col='{q.affected_column}'")
        print(f"  {q.question}")
    print()
    print("  (The orchestrator will ask these questions interactively when ask_clarifications=True)")
else:
    print("\nNo clarification questions needed — system can proceed autonomously.")

## 4. Run the full agentic pipeline

The orchestrator ties all agents together:
1. Optionally asks clarification questions interactively.
2. Runs the Planner → Executor → Evaluator loop for up to `max_rounds`.
3. Generates visualisations and a plain-language report.

Set `ask_clarifications=False` for non-interactive / batch use.

In [ ]:
result = run_agentic_pipeline(
    df,
    target             = TARGET,
    max_rounds         = 3,
    n_plans_per_round  = 3,
    cv                 = 5,
    score_threshold    = 0.90,
    verbose            = True,
    ask_clarifications = False,   # set True for interactive mode
    figures_dir        = "figures",
    generate_report    = True,
)

## 5. Results summary

In [ ]:
print(f"Run ID      : {result.run_id}")
print(f"Iterations  : {result.n_iterations}")
print(f"Converged   : {result.converged}")
print(f"Best score  : {result.best_result.score:.4f}")
print(f"CV std      : {result.best_result.cv_std:.4f}")
print(f"Metrics     : {result.best_result.metric_values}")
print(f"\nBest plan:")
print(f"  model              = {result.best_plan.model}")
print(f"  imputation         = {result.best_plan.imputation}")
print(f"  encoding           = {result.best_plan.encoding}")
print(f"  scaling            = {result.best_plan.scaling}")
print(f"  outlier_handling   = {result.best_plan.outlier_handling}")
print(f"  imbalance_strategy = {result.best_plan.imbalance_strategy}")

explanation = result.best_plan.model_params.get("__explanation", "")
if explanation:
    print(f"\nWhy this approach (from the LLM):\n  {explanation}")

## 6. Plain-language user report

The system generates a plain-language report designed for coaches, analysts, and scouts — no machine learning background required.

In [ ]:
from IPython.display import Markdown, display

if result.user_report:
    display(Markdown(result.user_report))
else:
    print("Report not generated (generate_report=False or an error occurred).")

## 7. Visualisations

The system saves a standard set of figures to `figures/`.  Display them inline:

In [ ]:
import os
from IPython.display import Image, display

figures_dir = "figures"
if os.path.isdir(figures_dir):
    pngs = sorted(f for f in os.listdir(figures_dir) if f.endswith(".png"))
    if pngs:
        for fname in pngs:
            print(f"\n── {fname} ──")
            display(Image(filename=os.path.join(figures_dir, fname)))
    else:
        print("No PNG figures found in figures/")
else:
    print("figures/ directory does not exist — run the pipeline first.")

## 8. Iteration history

In [ ]:
import pandas as pd

rows = []
for rec in result.history:
    rows.append({
        "iteration"  : rec.iteration,
        "model"      : rec.plan.model,
        "imputation" : rec.plan.imputation,
        "encoding"   : rec.plan.encoding,
        "scaling"    : rec.plan.scaling,
        "score"      : round(rec.result.score, 4),
        "cv_std"     : round(rec.result.cv_std, 4),
        **{k: round(v, 4) for k, v in rec.result.metric_values.items() if isinstance(v, float)},
    })

pd.DataFrame(rows).sort_values("score", ascending=False)

## 9. Use the best pipeline for prediction

In [ ]:
X_train, X_test, y_train, y_test = split_data(df, TARGET)
y_pred = result.best_pipeline.predict(X_test)

print(f"Test predictions (first 10): {y_pred[:10]}")

## 10. Inspect past runs (SQLite)

In [ ]:
from src.memory.run_store import RunStore
from src.orchestrator import _DEFAULT_DB_PATH

with RunStore(_DEFAULT_DB_PATH) as store:
    print("All run IDs stored so far:")
    for run_id in store.list_runs():
        attempts = store.load_attempts(run_id)
        best = max(attempts, key=lambda r: r.result.score)
        print(f"  {run_id[:8]}...  {len(attempts)} attempts  best_score={best.result.score:.4f}")

---
## 11. Ablation Study

Evaluates six variants and compares them side-by-side.  Matches the ablation design in the research proposal.

| Variant | Memory | Feedback loop | Clarification |
|---------|--------|---------------|---------------|
| 1. Rule-based baseline | — | — | — |
| 2. Search-based baseline (Optuna) | — | — | — |
| 3. Agentic — no memory | ✗ | ✓ | ✓ |
| 4. Agentic — no feedback (1 round) | ✓ | ✗ | ✓ |
| 5a/b. Warm-start experiment (run 1 + 2) | ✓ | ✓ | ✓ |
| **6. Full agentic system** | **✓** | **✓** | **✓** |

In [ ]:
from src.baselines.rule_based import run_rule_based
from src.baselines.search_based import run_search_based

ABLATION_TRIALS = 50
ABLATION_CV     = 5
ABLATION_ROUNDS = 3

In [ ]:
# --- Variant 1: Rule-based baseline ---
rb = run_rule_based(df, TARGET, cv=ABLATION_CV)
print(f"Rule-based  score={rb.score:.4f}  metrics={rb.metric_values}  runtime={rb.runtime_secs:.1f}s")

In [ ]:
# --- Variant 2: Search-based baseline (Optuna TPE) ---
sb = run_search_based(df, TARGET, n_trials=ABLATION_TRIALS, cv=ABLATION_CV)
print(f"Search-based  score={sb.score:.4f}  metrics={sb.metric_values}  runtime={sb.runtime_secs:.1f}s")

In [ ]:
# --- Variant 3: Agentic system without cross-run memory ---
ag_no_mem = run_agentic_pipeline(
    df, target=TARGET,
    max_rounds=ABLATION_ROUNDS, n_plans_per_round=3,
    cv=ABLATION_CV, score_threshold=0.90,
    use_memory=False, ask_clarifications=False,
    generate_report=False, verbose=True,
)
print(f"Agentic (no memory)  score={ag_no_mem.best_result.score:.4f}  "
      f"iterations={ag_no_mem.n_iterations}  metrics={ag_no_mem.best_result.metric_values}")

In [ ]:
# --- Variant 4: Agentic system without feedback loop (single round) ---
ag_no_fb = run_agentic_pipeline(
    df, target=TARGET,
    max_rounds=1,                  # ablation: one Planner call, no iterative refinement
    n_plans_per_round=3,
    cv=ABLATION_CV, score_threshold=0.90,
    use_memory=True, ask_clarifications=False,
    generate_report=False, verbose=True,
)
print(f"Agentic (no feedback)  score={ag_no_fb.best_result.score:.4f}  "
      f"iterations={ag_no_fb.n_iterations}  metrics={ag_no_fb.best_result.metric_values}")

In [ ]:
# --- Variant 5: Warm-start experiment ---
# Run twice on the same dataset. The second run retrieves the best plan from
# the first via ChromaDB.  Lower iteration count or higher score on run 2
# demonstrates the benefit of cross-run memory.
ag_warm1 = run_agentic_pipeline(
    df, target=TARGET, max_rounds=ABLATION_ROUNDS, n_plans_per_round=3,
    cv=ABLATION_CV, score_threshold=0.90,
    use_memory=True, ask_clarifications=False, generate_report=False, verbose=False,
)
ag_warm2 = run_agentic_pipeline(
    df, target=TARGET, max_rounds=ABLATION_ROUNDS, n_plans_per_round=3,
    cv=ABLATION_CV, score_threshold=0.90,
    use_memory=True, ask_clarifications=False, generate_report=False, verbose=False,
)
print(f"Warm-start run 1  score={ag_warm1.best_result.score:.4f}  iterations={ag_warm1.n_iterations}")
print(f"Warm-start run 2  score={ag_warm2.best_result.score:.4f}  iterations={ag_warm2.n_iterations}")
print(f"Δ score = {ag_warm2.best_result.score - ag_warm1.best_result.score:+.4f}  "
      f"Δ iterations = {ag_warm2.n_iterations - ag_warm1.n_iterations:+d}")

In [ ]:
import pandas as pd

def _row(variant, score, metric_values, cv_std, n_configs, n_iters, plan):
    row = {
        "variant":   variant,
        "score":     round(score, 4),
        "cv_std":    round(cv_std, 4),
        "n_configs": n_configs,
        "n_iters":   n_iters,
    }
    for k, v in metric_values.items():
        row[k] = round(v, 4) if isinstance(v, float) else v
    if plan is not None:
        row["model"]              = plan.model
        row["imputation"]         = plan.imputation
        row["encoding"]           = plan.encoding
        row["scaling"]            = plan.scaling
        row["outlier_handling"]   = plan.outlier_handling
        row["imbalance_strategy"] = plan.imbalance_strategy
    return row

rows = [
    _row("1_rule_based", rb.score, rb.metric_values, rb.cv_std,
         rb.n_configs_evaluated, "—", rb.best_plan),
    _row("2_search_based_optuna", sb.score, sb.metric_values, sb.cv_std,
         sb.n_configs_evaluated, "—", sb.best_plan),
    _row("3_agentic_no_memory",
         ag_no_mem.best_result.score, ag_no_mem.best_result.metric_values,
         ag_no_mem.best_result.cv_std, "—", ag_no_mem.n_iterations, ag_no_mem.best_plan),
    _row("4_agentic_no_feedback",
         ag_no_fb.best_result.score, ag_no_fb.best_result.metric_values,
         ag_no_fb.best_result.cv_std, "—", ag_no_fb.n_iterations, ag_no_fb.best_plan),
    _row("5a_warm_start_run1",
         ag_warm1.best_result.score, ag_warm1.best_result.metric_values,
         ag_warm1.best_result.cv_std, "—", ag_warm1.n_iterations, ag_warm1.best_plan),
    _row("5b_warm_start_run2",
         ag_warm2.best_result.score, ag_warm2.best_result.metric_values,
         ag_warm2.best_result.cv_std, "—", ag_warm2.n_iterations, ag_warm2.best_plan),
    _row("6_full_agentic",
         result.best_result.score, result.best_result.metric_values,
         result.best_result.cv_std, "—", result.n_iterations, result.best_plan),
]

results_df = pd.DataFrame(rows)
print("=== Full ablation results ===")
results_df

In [ ]:
import re, json
from pathlib import Path

dataset_slug = re.sub(r'[^\w]+', '_', TARGET).lower()
out_dir = Path('results') / dataset_slug
out_dir.mkdir(parents=True, exist_ok=True)

def _row_full(variant, score, metric_values, cv_std, n_configs,
              n_iters, plan, runtime=None, converged=None):
    row = {
        'dataset':   TARGET,
        'variant':   variant,
        'score':     round(score, 6),
        'cv_std':    round(cv_std, 6),
        'n_configs': n_configs,
        'n_iters':   n_iters,
        'converged': converged,
        'runtime_secs': round(runtime, 2) if runtime is not None else None,
    }
    for k, v in metric_values.items():
        row[k] = round(v, 6) if isinstance(v, float) else v
    if plan is not None:
        row['model']               = plan.model
        row['imputation']          = plan.imputation
        row['encoding']            = plan.encoding
        row['scaling']             = plan.scaling
        row['outlier_handling']    = plan.outlier_handling
        row['imbalance_strategy']  = plan.imbalance_strategy
        row['model_params']        = json.dumps({k: v for k, v in plan.model_params.items() if not k.startswith('__')})
    return row

summary_rows = [
    _row_full('1_rule_based', rb.score, rb.metric_values, rb.cv_std,
              rb.n_configs_evaluated, '—', rb.best_plan, rb.runtime_secs),
    _row_full('2_search_based_optuna', sb.score, sb.metric_values, sb.cv_std,
              sb.n_configs_evaluated, '—', sb.best_plan, sb.runtime_secs),
    _row_full('3_agentic_no_memory',
              ag_no_mem.best_result.score, ag_no_mem.best_result.metric_values,
              ag_no_mem.best_result.cv_std, '—', ag_no_mem.n_iterations,
              ag_no_mem.best_plan, converged=ag_no_mem.converged),
    _row_full('4_agentic_no_feedback',
              ag_no_fb.best_result.score, ag_no_fb.best_result.metric_values,
              ag_no_fb.best_result.cv_std, '—', ag_no_fb.n_iterations,
              ag_no_fb.best_plan, converged=ag_no_fb.converged),
    _row_full('5a_warm_start_run1',
              ag_warm1.best_result.score, ag_warm1.best_result.metric_values,
              ag_warm1.best_result.cv_std, '—', ag_warm1.n_iterations,
              ag_warm1.best_plan, converged=ag_warm1.converged),
    _row_full('5b_warm_start_run2',
              ag_warm2.best_result.score, ag_warm2.best_result.metric_values,
              ag_warm2.best_result.cv_std, '—', ag_warm2.n_iterations,
              ag_warm2.best_plan, converged=ag_warm2.converged),
    _row_full('6_full_agentic',
              result.best_result.score, result.best_result.metric_values,
              result.best_result.cv_std, '—', result.n_iterations,
              result.best_plan, converged=result.converged),
]
summary_df = pd.DataFrame(summary_rows)

history_rows = []
for variant_name, run_result in [
    ('3_agentic_no_memory',   ag_no_mem),
    ('4_agentic_no_feedback', ag_no_fb),
    ('5a_warm_start_run1',    ag_warm1),
    ('5b_warm_start_run2',    ag_warm2),
    ('6_full_agentic',        result),
]:
    for rec in run_result.history:
        row = {
            'dataset':    TARGET,
            'variant':    variant_name,
            'iteration':  rec.iteration,
            'plan_id':    rec.plan.plan_id,
            'score':      round(rec.result.score, 6),
            'cv_std':     round(rec.result.cv_std, 6),
            'runtime_secs': round(rec.result.runtime_secs, 2),
            'model':               rec.plan.model,
            'imputation':          rec.plan.imputation,
            'encoding':            rec.plan.encoding,
            'scaling':             rec.plan.scaling,
            'outlier_handling':    rec.plan.outlier_handling,
            'imbalance_strategy':  rec.plan.imbalance_strategy,
            'model_params':        json.dumps({k: v for k, v in rec.plan.model_params.items() if not k.startswith('__')}),
        }
        for k, v in rec.result.metric_values.items():
            row[k] = round(v, 6) if isinstance(v, float) else v
        history_rows.append(row)
history_df = pd.DataFrame(history_rows)

profile_row = {
    'dataset':         TARGET,
    'n_rows':          profile.n_rows,
    'n_cols':          profile.n_cols,
    'target_type':     profile.target_type.value,
    'n_duplicates':    profile.n_duplicates,
    'class_distribution': json.dumps(profile.class_distribution),
    'n_issues':        len(issues),
    'n_issues_high':   sum(i.severity.value == 'high' for i in issues),
    'n_issues_medium': sum(i.severity.value == 'medium' for i in issues),
    'n_issues_low':    sum(i.severity.value == 'low' for i in issues),
    'issue_types':     json.dumps([i.issue_type.value for i in issues]),
    'n_clusters':      profile.clusters.n_clusters if profile.clusters else 0,
    'silhouette_score': profile.clusters.silhouette_score if profile.clusters else None,
    'overall_missing_rate': round(
        sum(c.missing_rate for c in profile.columns.values()) / len(profile.columns), 4),
}
profile_df = pd.DataFrame([profile_row])

warmstart_df = pd.DataFrame([{
    'dataset':   TARGET,
    'run':       'run_1_cold',
    'score':     round(ag_warm1.best_result.score, 6),
    'cv_std':    round(ag_warm1.best_result.cv_std, 6),
    'n_iters':   ag_warm1.n_iterations,
    'converged': ag_warm1.converged,
    'model':     ag_warm1.best_plan.model,
    **{k: round(v, 6) for k, v in ag_warm1.best_result.metric_values.items()},
}, {
    'dataset':   TARGET,
    'run':       'run_2_warm',
    'score':     round(ag_warm2.best_result.score, 6),
    'cv_std':    round(ag_warm2.best_result.cv_std, 6),
    'n_iters':   ag_warm2.n_iterations,
    'converged': ag_warm2.converged,
    'model':     ag_warm2.best_plan.model,
    **{k: round(v, 6) for k, v in ag_warm2.best_result.metric_values.items()},
}])
warmstart_df['delta_score'] = (
    warmstart_df.loc[warmstart_df.run == 'run_2_warm', 'score'].values[0] -
    warmstart_df.loc[warmstart_df.run == 'run_1_cold', 'score'].values[0]
)
warmstart_df['delta_iters'] = (
    warmstart_df.loc[warmstart_df.run == 'run_2_warm', 'n_iters'].values[0] -
    warmstart_df.loc[warmstart_df.run == 'run_1_cold', 'n_iters'].values[0]
)

files = {
    'ablation_summary.csv':  summary_df,
    'iteration_history.csv': history_df,
    'dataset_profile.csv':   profile_df,
    'warmstart.csv':         warmstart_df,
}
for fname, data in files.items():
    path = out_dir / fname
    data.to_csv(path, index=False)
    print(f'Saved {path}  ({len(data)} rows × {len(data.columns)} cols)')

print(f'\nAll results saved to: {out_dir}/')
summary_df